In [3]:
# Cell 1: Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-1000:] if result.stdout else "")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
    return result.returncode == 0

run("pip install timm")
run("pip install faiss-cpu")
print("✅ Dependencies ready")

Mounted at /content/drive
ady satisfied: click>=8.2.1 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub->timm) (8.3.1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.6 MB/s eta 0:00:00

✅ Dependencies ready


In [6]:
# Cell 2: Unzip datasets and verify
import os, subprocess
from pathlib import Path

DRIVE_ROOT  = "/content/drive/MyDrive/reid_project"
LOCAL_DATA  = "/content/data"
MARKET_ROOT = f"{LOCAL_DATA}/Market-1501-v15.09.15"
DUKE_ROOT   = f"{LOCAL_DATA}/DukeMTMC-reID"

os.makedirs(LOCAL_DATA, exist_ok=True)

# ── Unzip ──────────────────────────────────────────────────
def unzip_if_needed(zip_path, extract_to, check_folder):
    if os.path.exists(check_folder):
        print(f"✓ Already extracted: {check_folder}")
        return
    print(f"Unzipping {zip_path} ...")
    result = subprocess.run(
        f"unzip -q '{zip_path}' -d '{extract_to}'",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    else:
        print(f"✓ Done: {check_folder}")

unzip_if_needed(
    f"{DRIVE_ROOT}/Market-1501.zip",
    LOCAL_DATA,
    MARKET_ROOT
)
unzip_if_needed(
    f"{DRIVE_ROOT}/DukeMTMC-reID.zip",
    LOCAL_DATA,
    DUKE_ROOT
)

# ── Verify ─────────────────────────────────────────────────
def verify(root, name):
    folders = ["bounding_box_train", "bounding_box_test", "query"]
    print(f"\n[{name}] {root}")
    ok = True
    for f in folders:
        p = Path(root) / f
        if p.exists():
            print(f"  ✓ {f}: {len(list(p.glob('*.jpg')))} images")
        else:
            print(f"  ✗ MISSING: {f}")
            ok = False
    return ok

verify(MARKET_ROOT, "Market-1501")
verify(DUKE_ROOT, "DukeMTMC-reID")

✓ Already extracted: /content/data/Market-1501-v15.09.15
Unzipping /content/drive/MyDrive/reid_project/DukeMTMC-reID.zip ...
ERROR: replace /content/data/bounding_box_test/0002_c1_f0044158.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename:  NULL
(EOF or read error, treating as "[N]one" ...)


[Market-1501] /content/data/Market-1501-v15.09.15
  ✓ bounding_box_train: 12936 images
  ✓ bounding_box_test: 19732 images
  ✓ query: 3368 images

[DukeMTMC-reID] /content/data/DukeMTMC-reID
  ✗ MISSING: bounding_box_train
  ✗ MISSING: bounding_box_test
  ✗ MISSING: query


False

In [7]:
# Fix: Duke zip extracts flat, create parent folder and move
import os, subprocess

LOCAL_DATA = "/content/data"
DUKE_ROOT  = f"{LOCAL_DATA}/DukeMTMC-reID"

# Force unzip overwriting everything
subprocess.run(
    f"unzip -o -q '/content/drive/MyDrive/reid_project/DukeMTMC-reID.zip' -d '{LOCAL_DATA}/duke_tmp'",
    shell=True
)

# Check what was extracted
result = subprocess.run(f"ls {LOCAL_DATA}/duke_tmp/", shell=True, capture_output=True, text=True)
print("Extracted:", result.stdout)

# Move into correct structure
os.makedirs(DUKE_ROOT, exist_ok=True)
for folder in ["bounding_box_train", "bounding_box_test", "query"]:
    src = f"{LOCAL_DATA}/duke_tmp/{folder}"
    dst = f"{DUKE_ROOT}/{folder}"
    if os.path.exists(src) and not os.path.exists(dst):
        os.rename(src, dst)
        print(f"✓ Moved {folder}")
    elif os.path.exists(dst):
        print(f"✓ Already exists: {folder}")
    else:
        print(f"✗ Not found: {folder}")

# Verify
from pathlib import Path
for folder in ["bounding_box_train", "bounding_box_test", "query"]:
    p = Path(DUKE_ROOT) / folder
    count = len(list(p.glob("*.jpg"))) if p.exists() else 0
    print(f"  {folder}: {count} images")

Extracted: bounding_box_test
bounding_box_train
query

✓ Moved bounding_box_train
✓ Moved bounding_box_test
✓ Moved query
  bounding_box_train: 16522 images
  bounding_box_test: 17661 images
  query: 2228 images


In [8]:
# Cell 3: Config
import torch

# ── Paths ──────────────────────────────────────────────────
LOCAL_DATA  = "/content/data"
DRIVE_ROOT     = "/content/drive/MyDrive/reid_project"
MARKET_ROOT = f"{LOCAL_DATA}/Market-1501-v15.09.15"
DUKE_ROOT   = f"{LOCAL_DATA}/DukeMTMC-reID"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUTPUT_DIR     = "/content/reid_output"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────
BACKBONE       = "resnet50_ibn_a"
FEAT_DIM       = 2048
NUM_CLASSES    = 751        # Market-1501 training IDs

# ── Training ───────────────────────────────────────────────
NUM_EPOCHS     = 60
WARMUP_EPOCHS  = 10
BATCH_SIZE     = 64
NUM_INSTANCE   = 4          # images per ID per batch
BASE_LR        = 0.00035
WEIGHT_DECAY   = 0.0005
LABEL_SMOOTH   = 0.1

# ── Augmentation ───────────────────────────────────────────
INPUT_H, INPUT_W      = 256, 128
RANDOM_ERASING_PROB   = 0.5
NORMALIZE_MEAN        = [0.485, 0.456, 0.406]
NORMALIZE_STD         = [0.229, 0.224, 0.225]

# ── Misc ───────────────────────────────────────────────────
SEED           = 42
NUM_WORKERS    = 2
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOG_PERIOD     = 100        # iterations
CKPT_PERIOD    = 10         # epochs

print(f"✅ Config ready | device: {DEVICE}")

✅ Config ready | device: cuda


In [9]:
# Cell 4: Verify GPU
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


In [10]:
# Cell 5: Dataset and DataLoader
import os, re, torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from collections import defaultdict

# ── Parse Market-1501 / DukeMTMC image names ──────────────
def parse_reid_folder(folder):
    """Returns list of (img_path, pid, camid)"""
    samples = []
    pid_set = set()
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith(".jpg"):
            continue
        if fname.startswith("-1") or fname.startswith("0000"):
            continue                         # junk / distractor images
        pid  = int(fname.split("_")[0])
        cam  = int(re.search(r'c(\d+)', fname).group(1)) - 1
        samples.append((os.path.join(folder, fname), pid, cam))
        pid_set.add(pid)
    # Remap pids to 0-indexed contiguous range
    pid_map = {p: i for i, p in enumerate(sorted(pid_set))}
    samples = [(p, pid_map[pid], cam) for p, pid, cam in samples]
    return samples

# ── ReID Dataset ───────────────────────────────────────────
class ReIDDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples  = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, pid, camid = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, camid

# ── Random Identity Sampler ────────────────────────────────
class RandomIdentitySampler(Sampler):
    """Sample NUM_INSTANCE images per identity per batch."""
    def __init__(self, samples, batch_size, num_instance):
        self.batch_size    = batch_size
        self.num_instance  = num_instance
        self.num_pids_per_batch = batch_size // num_instance

        pid_to_idx = defaultdict(list)
        for idx, (_, pid, _) in enumerate(samples):
            pid_to_idx[pid].append(idx)
        self.pid_to_idx = {p: np.array(idxs) for p, idxs in pid_to_idx.items()}
        self.pids = list(self.pid_to_idx.keys())

    def __len__(self):
        return len(self.pids) * self.num_instance

    def __iter__(self):
        pids = np.random.permutation(self.pids)
        batch, final = [], []
        for pid in pids:
            idxs = self.pid_to_idx[pid]
            replace = len(idxs) < self.num_instance
            chosen  = np.random.choice(idxs, self.num_instance, replace=replace)
            batch.extend(chosen.tolist())
            if len(batch) == self.batch_size:
                final.extend(batch)
                batch = []
        return iter(final)

# ── Transforms ─────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((INPUT_H, INPUT_W)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
    transforms.RandomErasing(p=RANDOM_ERASING_PROB),
])

test_tf = transforms.Compose([
    transforms.Resize((INPUT_H, INPUT_W)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

# ── Build loaders ──────────────────────────────────────────
def build_loaders(dataset_root):
    train_samples   = parse_reid_folder(f"{dataset_root}/bounding_box_train")
    query_samples   = parse_reid_folder(f"{dataset_root}/query")
    gallery_samples = parse_reid_folder(f"{dataset_root}/bounding_box_test")

    train_ds   = ReIDDataset(train_samples, train_tf)
    query_ds   = ReIDDataset(query_samples, test_tf)
    gallery_ds = ReIDDataset(gallery_samples, test_tf)

    sampler = RandomIdentitySampler(train_samples, BATCH_SIZE, NUM_INSTANCE)

    train_loader   = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                sampler=sampler, num_workers=NUM_WORKERS,
                                pin_memory=True, drop_last=True)
    query_loader   = DataLoader(query_ds, batch_size=128, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)
    gallery_loader = DataLoader(gallery_ds, batch_size=128, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

    num_classes = len(set(pid for _, pid, _ in train_samples))
    print(f"  train  : {len(train_samples)} images, {num_classes} IDs")
    print(f"  query  : {len(query_samples)} images")
    print(f"  gallery: {len(gallery_samples)} images")
    return train_loader, query_loader, gallery_loader, num_classes

print("[Market-1501]")
train_loader, query_loader, gallery_loader, NUM_CLASSES = build_loaders(MARKET_ROOT)
print(f"✅ DataLoaders ready | {NUM_CLASSES} training IDs")

[Market-1501]
  train  : 12936 images, 751 IDs
  query  : 3368 images
  gallery: 13115 images
✅ DataLoaders ready | 751 training IDs


In [11]:
# Cell 6: Model — ResNet-50 with IBN-a + BNNeck
import torch, torch.nn as nn
import timm

# ── IBN-a layer ────────────────────────────────────────────
class IBN(nn.Module):
    """Instance-Batch Normalization (IBN-a).
    Splits channels: half IN, half BN — improves domain generalization.
    """
    def __init__(self, channels):
        super().__init__()
        half = channels // 2
        self.IN = nn.InstanceNorm2d(half, affine=True)
        self.BN = nn.BatchNorm2d(half)

    def forward(self, x):
        half = x.size(1) // 2
        x_in = self.IN(x[:, :half])
        x_bn = self.BN(x[:, half:])
        return torch.cat([x_in, x_bn], dim=1)

def add_ibn_a(model):
    """Patch IBN-a into layer1, layer2, layer3 of a ResNet-50.
    Replaces the first BN (bn1) in each bottleneck block with IBN.
    layer4 is left unchanged (standard BN) — standard IBN-a recipe.
    """
    for layer_name in ["layer1", "layer2", "layer3"]:
        layer = getattr(model, layer_name)
        for block in layer:
            # Bottleneck blocks have bn1 after the 1x1 conv
            if hasattr(block, "bn1"):
                c = block.bn1.num_features
                block.bn1 = IBN(c)
    return model

# ── BNNeck ─────────────────────────────────────────────────
class BNNeck(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.bn = nn.BatchNorm1d(feat_dim)
        self.bn.bias.requires_grad_(False)

    def forward(self, x):
        return self.bn(x)

# ── Full ReID Model ────────────────────────────────────────
class ReIDModel(nn.Module):
    def __init__(self, num_classes, feat_dim=2048):
        super().__init__()
        # Load pretrained ResNet-50, strip classifier
        backbone = timm.create_model("resnet50", pretrained=True,
                                     num_classes=0, global_pool="avg")
        # Patch IBN-a into layers 1-3
        self.backbone   = add_ibn_a(backbone)
        self.feat_dim   = feat_dim
        self.bnneck     = BNNeck(feat_dim)
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward(self, x):
        feat    = self.backbone(x)               # (B, 2048)
        feat_bn = self.bnneck(feat)              # (B, 2048) BN normalized
        if self.training:
            logits = self.classifier(feat_bn)
            return logits, feat                  # raw feat for triplet
        return nn.functional.normalize(feat_bn, dim=1)  # L2 norm for retrieval

model = ReIDModel(NUM_CLASSES, FEAT_DIM).to(DEVICE)
print(f"✅ Model ready | params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Sanity check
with torch.no_grad():
    dummy = torch.randn(4, 3, INPUT_H, INPUT_W).to(DEVICE)
    model.train()
    logits, feat = model(dummy)
    print(f"   logits : {logits.shape}")
    print(f"   feat   : {feat.shape}")
    model.eval()
    emb = model(dummy)
    print(f"   embed  : {emb.shape}")
    print(f"   norm   : {emb.norm(dim=1).mean():.4f} (should be ~1.0)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

✅ Model ready | params: 25.1M
   logits : torch.Size([4, 751])
   feat   : torch.Size([4, 2048])
   embed  : torch.Size([4, 2048])
   norm   : 1.0000 (should be ~1.0)


In [12]:
# Cell 7: Loss and Optimizer
import torch, torch.nn as nn
import math

# ── Label Smoothing Cross Entropy ─────────────────────────
class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon     = epsilon
        self.logsoftmax  = nn.LogSoftmax(dim=1)

    def forward(self, logits, targets):
        log_probs = self.logsoftmax(logits)
        # smooth targets
        with torch.no_grad():
            smooth = torch.full_like(log_probs,
                                     self.epsilon / (self.num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.epsilon)
        return (-smooth * log_probs).sum(dim=1).mean()

# ── Triplet Loss with hard mining ─────────────────────────
class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, feats, pids):
        # L2 normalize
        feats = nn.functional.normalize(feats, dim=1)
        dist  = torch.cdist(feats, feats, p=2)

        mask_pos = pids.unsqueeze(1) == pids.unsqueeze(0)
        mask_neg = ~mask_pos

        # Hard positive: max dist among same ID
        ap = (dist * mask_pos.float()).max(dim=1)[0]
        # Hard negative: min dist among different ID
        dist_neg = dist.clone()
        dist_neg[mask_neg == 0] = float('inf')
        an = dist_neg.min(dim=1)[0]

        loss = torch.clamp(ap - an + self.margin, min=0.0)
        return loss.mean()

# ── Optimizer ─────────────────────────────────────────────
criterion_ce      = LabelSmoothingCE(NUM_CLASSES, LABEL_SMOOTH).to(DEVICE)
criterion_triplet = TripletLoss(margin=0.3).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY
)

# ── LR scheduler: linear warmup + cosine decay ────────────
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(NUM_EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("✅ Loss functions, optimizer, scheduler ready")
print(f"   CE label smooth : {LABEL_SMOOTH}")
print(f"   Triplet margin  : 0.3")
print(f"   Warmup epochs   : {WARMUP_EPOCHS}")

✅ Loss functions, optimizer, scheduler ready
   CE label smooth : 0.1
   Triplet margin  : 0.3
   Warmup epochs   : 10


In [13]:
# Cell 8: CMC and mAP evaluation
import torch
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    embeddings, pids, camids = [], [], []
    for imgs, pid, camid in tqdm(loader, desc="Extracting", leave=False):
        imgs = imgs.to(device)
        emb  = model(imgs)
        embeddings.append(emb.cpu())
        pids.extend(pid.tolist())
        camids.extend(camid.tolist())
    return torch.cat(embeddings), np.array(pids), np.array(camids)

def eval_reid(query_emb, query_pids, query_cams,
              gallery_emb, gallery_pids, gallery_cams,
              max_rank=10):
    # Cosine similarity via dot product on L2-normed embeddings
    sim   = torch.mm(query_emb, gallery_emb.t()).numpy()
    n_q   = len(query_pids)
    cmc   = np.zeros(max_rank)
    ap_sum = 0.0

    for q in range(n_q):
        order   = np.argsort(-sim[q])
        g_pids  = gallery_pids[order]
        g_cams  = gallery_cams[order]

        # Remove same camera same ID (junk)
        keep = ~((g_pids == query_pids[q]) & (g_cams == query_cams[q]))
        g_pids = g_pids[keep]

        # Matches
        matches = (g_pids == query_pids[q]).astype(np.float32)

        # CMC
        for r in range(max_rank):
            if matches[:r+1].sum() > 0:
                cmc[r:] += 1
                break

        # AP
        num_rel   = matches.sum()
        if num_rel == 0:
            continue
        cum_match = np.cumsum(matches)
        precision = cum_match / (np.arange(len(matches)) + 1)
        ap_sum   += (precision * matches).sum() / num_rel

    cmc    = cmc / n_q
    mAP    = ap_sum / n_q
    return cmc, mAP

def run_evaluation(model, q_loader, g_loader, device, label=""):
    q_emb, q_pids, q_cams = extract_embeddings(model, q_loader, device)
    g_emb, g_pids, g_cams = extract_embeddings(model, g_loader, device)
    cmc, mAP = eval_reid(q_emb, q_pids, q_cams, g_emb, g_pids, g_cams)
    print(f"\n── {label} Results ─────────────────────")
    print(f"  Rank-1 : {cmc[0]*100:.2f}%")
    print(f"  Rank-5 : {cmc[4]*100:.2f}%")
    print(f"  Rank-10: {cmc[9]*100:.2f}%")
    print(f"  mAP    : {mAP*100:.2f}%")
    return cmc, mAP

print("✅ Evaluation functions ready")

✅ Evaluation functions ready


In [14]:
# Cell 9: Training loop
import torch, os, shutil
from tqdm import tqdm

best_mAP   = 0.0
best_ckpt  = None

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = ce_loss = tri_loss = 0.0
    n_batches  = 0

    for imgs, pids, _ in tqdm(train_loader,
                               desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
                               leave=False):
        imgs = imgs.to(DEVICE)
        pids = pids.to(DEVICE)

        logits, feats = model(imgs)

        loss_ce  = criterion_ce(logits, pids)
        loss_tri = criterion_triplet(feats, pids)
        loss     = loss_ce + loss_tri

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        ce_loss    += loss_ce.item()
        tri_loss   += loss_tri.item()
        n_batches  += 1

    scheduler.step()

    avg_total = total_loss / n_batches
    avg_ce    = ce_loss    / n_batches
    avg_tri   = tri_loss   / n_batches
    lr_now    = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"loss: {avg_total:.4f} (ce: {avg_ce:.4f}, tri: {avg_tri:.4f}) | "
          f"lr: {lr_now:.2e}")

    # ── Evaluate every 10 epochs ──────────────────────────
    if (epoch + 1) % 10 == 0:
        cmc, mAP = run_evaluation(
            model, query_loader, gallery_loader,
            DEVICE, f"Market-1501 Epoch {epoch+1}"
        )
        # Save checkpoint
        ckpt_path = f"{OUTPUT_DIR}/model_epoch{epoch+1}.pth"
        torch.save({
            "epoch":      epoch + 1,
            "state_dict": model.state_dict(),
            "mAP":        mAP,
            "cmc":        cmc,
        }, ckpt_path)
        print(f"  💾 Saved: {ckpt_path}")

        # Track best
        if mAP > best_mAP:
            best_mAP  = mAP
            best_ckpt = ckpt_path
            shutil.copy(ckpt_path, f"{OUTPUT_DIR}/model_best.pth")
            print(f"  ⭐ New best mAP: {best_mAP*100:.2f}%")

        model.train()

print(f"\n✅ Training complete | best mAP: {best_mAP*100:.2f}%")

Epoch   1/60 | loss: 7.0477 (ce: 6.6224, tri: 0.4253) | lr: 7.00e-05


Epoch   2/60 | loss: 7.0232 (ce: 6.6033, tri: 0.4199) | lr: 1.05e-04


Epoch   3/60 | loss: 6.9769 (ce: 6.5620, tri: 0.4150) | lr: 1.40e-04


Epoch   4/60 | loss: 6.9007 (ce: 6.4903, tri: 0.4104) | lr: 1.75e-04


Epoch   5/60 | loss: 6.7781 (ce: 6.3735, tri: 0.4047) | lr: 2.10e-04


...

Epoch  57/60 | loss: 1.3287 (ce: 1.1839, tri: 0.1447) | lr: 3.10e-06


Epoch  58/60 | loss: 1.3372 (ce: 1.1927, tri: 0.1445) | lr: 1.38e-06


Epoch  59/60 | loss: 1.3440 (ce: 1.1947, tri: 0.1492) | lr: 3.45e-07


Epoch  60/60 | loss: 1.3323 (ce: 1.1864, tri: 0.1459) | lr: 0.00e+00



── Market-1501 Epoch 60 Results ─────────────────────
  Rank-1 : 81.80%
  Rank-5 : 92.64%
  Rank-10: 95.64%
  mAP    : 60.38%
  💾 Saved: /content/reid_output/model_epoch60.pth
  ⭐ New best mAP: 60.38%

✅ Training complete | best mAP: 60.38%


In [15]:
# Cell 10: Save best checkpoint to Drive
import shutil, os

best_local = f"{OUTPUT_DIR}/model_best.pth"
dest       = f"{DRIVE_CKPT_DIR}/reid_best.pth"

if os.path.exists(best_local):
    shutil.copy(best_local, dest)
    print(f"✅ Best checkpoint saved to Drive: {dest}")
else:
    print("❌ No checkpoint found")

✅ Best checkpoint saved to Drive: /content/drive/MyDrive/reid_project/checkpoints/reid_best.pth


In [17]:
# Cell 11: Final evaluation on Market-1501
import numpy as np
import torch

# Fix 1: allow numpy globals in checkpoint
torch.serialization.add_safe_globals([np.ndarray, np.dtype])

# Fix 2: load with weights_only=False (we trust our own checkpoint)
ckpt = torch.load(f"{OUTPUT_DIR}/model_best.pth",
                  map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (mAP: {ckpt['mAP']*100:.2f}%)")

run_evaluation(model, query_loader, gallery_loader, DEVICE, "Market-1501 FINAL")

print("""
╔══════════════════════════════════════════════════════════╗
║         Training complete — next steps on M1             ║
╠══════════════════════════════════════════════════════════╣
║  1. Open Google Drive in browser                         ║
║  2. Navigate to: MyDrive/reid_project/checkpoints/       ║
║  3. Download: reid_best.pth                              ║
║  4. Move to your project:                                ║
║     mv ~/Downloads/reid_best.pth                         ║
║        ~/Desktop/Portfolio Projects/                     ║
║           person-reidentification/checkpoints/           ║
╚══════════════════════════════════════════════════════════╝
""")

Loaded best checkpoint from epoch 60 (mAP: 60.38%)



── Market-1501 FINAL Results ─────────────────────
  Rank-1 : 81.80%
  Rank-5 : 92.64%
  Rank-10: 95.64%
  mAP    : 60.38%

╔══════════════════════════════════════════════════════════╗
║         Training complete — next steps on M1             ║
╠══════════════════════════════════════════════════════════╣
║  1. Open Google Drive in browser                         ║
║  2. Navigate to: MyDrive/reid_project/checkpoints/       ║
║  3. Download: reid_best.pth                              ║
║  4. Move to your project:                                ║
║     mv ~/Downloads/reid_best.pth                         ║
║        ~/Desktop/Portfolio Projects/                     ║
║           person-reidentification/checkpoints/           ║
╚══════════════════════════════════════════════════════════╝

